# Noise robustness of PSR Z-Score vs Raw Z-Score under AWGN (Supplementary Table S7)

AWGN injection at SNR = +10, +5, 0, -5, -10 dB. Two experiments: A (clean train, noisy test) and B (matched noise). Three seeds.

**Paper:** Zafar SA, Qin W. *Cross-task anomaly detection in reconfigurable industrial robot systems based on physics-structured regression of joint motor currents* (2026).


In [ ]:
import os, glob, time, warnings
import numpy as np
import pandas as pd
import h5py
import scipy.stats as sst
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
np.random.seed(42)

# Paths and constants
ROOT = r"D:\Research\RCM"
BASE = os.path.join(ROOT, "Lab_Data")
OUT  = os.path.join(ROOT, "Processed_Data")
os.makedirs(OUT, exist_ok=True)
OUT_CSV = os.path.join(OUT, "NB17_R12_noise_robustness_zscore_pair.csv")

TASK_PAYLOAD = {"T1": 0.0, "T2": 1.0, "T3": 3.0, "T4": 2.0}
PAYLOAD_COM  = np.array([0.0, 0.0, 0.05])
GRAVITY      = np.array([0.0, 0.0, -9.81])
RATE         = 125
SUBSAMPLE    = 4
MIN_SAMP     = 200
N_BOOT       = 1000

UR5_DH_A     = [0, -0.42500, -0.39225, 0, 0, 0]
UR5_DH_D     = [0.089159, 0, 0, 0.10915, 0.09465, 0.0823]
UR5_DH_ALPHA = [np.pi/2, 0, 0, np.pi/2, -np.pi/2, 0]
UR5_MASS     = [3.7000, 8.3930, 2.2750, 1.2190, 1.2190, 0.1879]
UR5_COM      = [
    [0.0,     -0.02561,  0.00193],
    [0.21250,  0.0,      0.11336],
    [0.11993,  0.0,      0.02650],
    [0.0,     -0.00180,  0.01634],
    [0.0,      0.00180,  0.01634],
    [0.0,      0.0,     -0.00116],
]

V5_TABLE4 = {
    "PSR-ZScore":  {"T1": 0.966, "T2": 0.904, "T3": 0.648, "T4": 0.851, "mean": 0.842},
    "Raw-ZScore":  {"T1": 0.792, "T2": 0.025, "T3": 0.211, "T4": 0.850, "mean": 0.470},
}
ANCHOR_TOL = 0.005

# DH gravity torque
def dh_transform(a, d, alpha, theta):
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([
        [ct, -st*ca,  st*sa, a*ct],
        [st,  ct*ca, -ct*sa, a*st],
        [0,    sa,    ca,    d   ],
        [0,    0,     0,     1   ]
    ])

def gravity_torque(q, payload_mass=0.0):
    T = [np.eye(4)]
    for i in range(6):
        T.append(T[-1] @ dh_transform(UR5_DH_A[i], UR5_DH_D[i], UR5_DH_ALPHA[i], q[i]))
    com_world = [(T[i+1] @ np.array([*UR5_COM[i], 1.0]))[:3] for i in range(6)]
    masses = list(UR5_MASS)
    if payload_mass > 0:
        masses.append(payload_mass)
        com_world.append((T[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3])
    tau_g = np.zeros(6)
    dq = 1e-6
    for i in range(6):
        qp = q.copy(); qp[i] += dq
        Tp = [np.eye(4)]
        for jj in range(6):
            Tp.append(Tp[-1] @ dh_transform(UR5_DH_A[jj], UR5_DH_D[jj], UR5_DH_ALPHA[jj], qp[jj]))
        for jj in range(len(masses)):
            cp = ((Tp[jj+1] @ np.array([*UR5_COM[jj], 1.0]))[:3] if jj < 6
                  else (Tp[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3])
            tau_g[i] += masses[jj] * GRAVITY @ ((cp - com_world[jj]) / dq)
    return tau_g

REGISTRY = {
    "T1_healthy":   ("T1_PickPlace/Healthy",   "UR5_T1_healthy_180cyc_*.h5",                  "T1", 0),
    "T2_healthy":   ("T2_Assembly/Healthy",    "UR5_T2_healthy_180cyc_*.h5",                  "T2", 0),
    "T3_healthy":   ("T3_Palletize/Healthy",   "UR5_T3_healthy_183cyc_*.h5",                  "T3", 0),
    "T4_healthy":   ("T4_BinReorient/healthy", "UR5_T4_healthy_session2_35cyc_*.h5",          "T4", 0),
    "T1_A2_0.5kg":  ("T1_PickPlace/A2", "UR5_T1_A2_0.5kg_gripper_40cyc_*.h5",                 "T1", 1),
    "T1_A2_1kg":    ("T1_PickPlace/A2", "UR5_T1_A2_1kg_gripper_40cyc_*.h5",                   "T1", 1),
    "T1_A2_2kg":    ("T1_PickPlace/A2", "UR5_T1_A2_2kg_gripper_40cyc_*.h5",                   "T1", 1),
    "T1_A3_10wraps":("T1_PickPlace/A3", "UR5_T1_A3_1band_40cyc_*.h5",                         "T1", 1),
    "T1_A3_17wraps":("T1_PickPlace/A3", "UR5_T1_A3_3bands_40cyc_*.h5",                        "T1", 1),
    "T1_A5_20mm":   ("T1_PickPlace/A5", "UR5_T1_A5_20mm_40cyc_*.h5",                          "T1", 1),
    "T1_A5_50mm":   ("T1_PickPlace/A5", "UR5_T1_A5_50mm_40cyc_*.h5",                          "T1", 1),
    "T1_A5_100mm":  ("T1_PickPlace/A5", "UR5_T1_A5_100mm_40cyc_*.h5",                         "T1", 1),
    "T2_A2_1.5kg":  ("T2_Assembly/A2", "UR5_T2_A2_1.5kg_gripper_40cyc_*.h5",                  "T2", 1),
    "T2_A2_2kg":    ("T2_Assembly/A2", "UR5_T2_A2_2kg_gripper_40cyc_*.h5",                    "T2", 1),
    "T2_A2_3kg":    ("T2_Assembly/A2", "UR5_T2_A2_3kg_gripper_40cyc_*.h5",                    "T2", 1),
    "T2_A3_7duct":  ("T2_Assembly/A3", "UR5_T2_A3_light_duct_40cyc_*.h5",                     "T2", 1),
    "T2_A3_14duct": ("T2_Assembly/A3", "UR5_T2_A3_medium_duct_40cyc_*_225508.h5",             "T2", 1),
    "T2_A5_20mm":   ("T2_Assembly/A5", "UR5_T2_A5_20mm_40cyc_*.h5",                           "T2", 1),
    "T2_A5_50mm":   ("T2_Assembly/A5", "UR5_T2_A5_50mm_40cyc_*.h5",                           "T2", 1),
    "T2_A5_100mm":  ("T2_Assembly/A5", "UR5_T2_A5_100mm_40cyc_*.h5",                          "T2", 1),
    "T3_A2_3.5kg":  ("T3_Palletize/A2", "UR5_T3_A2_3.5kg_gripper_33cyc_*.h5",                 "T3", 1),
    "T3_A2_4kg":    ("T3_Palletize/A2", "UR5_T3_A2_4kg_gripper_33cyc_*.h5",                   "T3", 1),
    "T3_A2_5kg":    ("T3_Palletize/A2", "UR5_T3_A2_4.5kg_gripper_33cyc_*.h5",                 "T3", 1),
    "T3_A3_7duct":  ("T3_Palletize/A3", "UR5_T3_A3_light_duct_33cyc_*_222457.h5",             "T3", 1),
    "T3_A3_14duct": ("T3_Palletize/A3", "UR5_T3_A3_medium_duct_33cyc_*_205648.h5",            "T3", 1),
    "T3_A5_20mm":   ("T3_Palletize/A5", "UR5_T3_A5_20mm_33cyc_*_172334.h5",                   "T3", 1),
    "T3_A5_50mm":   ("T3_Palletize/A5", "UR5_T3_A5_50mm_33cyc_*_164447.h5",                   "T3", 1),
    "T3_A5_100mm":  ("T3_Palletize/A5", "UR5_T3_A5_100mm_33cyc_*_160716.h5",                  "T3", 1),
    "T4_A2_0.5kg":  ("T4_BinReorient/anomaly", "UR5_T4_A2_0.5kg_35cyc_*.h5",                  "T4", 1),
    "T4_A2_1kg":    ("T4_BinReorient/anomaly", "UR5_T4_A2_1kg_35cyc_*.h5",                    "T4", 1),
    "T4_A2_2kg":    ("T4_BinReorient/anomaly", "UR5_T4_A2_2kg_35cyc_*.h5",                    "T4", 1),
    "T4_A3_7duct":  ("T4_BinReorient/anomaly", "UR5_T4_A3_7wraps_35cyc_*.h5",                 "T4", 1),
    "T4_A3_14duct": ("T4_BinReorient/anomaly", "UR5_T4_A3_14wraps_35cyc_*.h5",                "T4", 1),
    "T4_A5_20mm":   ("T4_BinReorient/anomaly", "UR5_T4_A5_20mm_35cyc_*.h5",                   "T4", 1),
    "T4_A5_50mm":   ("T4_BinReorient/anomaly", "UR5_T4_A5_50mm_35cyc_*.h5",                   "T4", 1),
    "T4_A5_100mm":  ("T4_BinReorient/anomaly", "UR5_T4_A5_100mm_35cyc_*.h5",                  "T4", 1),
}

def load_all_cycles():
    all_cycles = []
    for key, (subdir, pattern, task, is_anom) in REGISTRY.items():
        matches = sorted(glob.glob(os.path.join(BASE, subdir, pattern)))
        if not matches:
            print(f"  WARNING  Not found: {key}")
            continue
        for path in matches:
            with h5py.File(path, "r") as f:
                cnum   = f["cycle_number"][:].astype(int).ravel()
                cur_a  = f["actual_current"][:]
                q_a    = f["actual_q"][:]
                qd_a   = f["actual_qd"][:]
            for c in np.unique(cnum[cnum > 0]):
                mask = cnum == c
                if mask.sum() >= MIN_SAMP:
                    all_cycles.append({
                        "q":              q_a[mask],
                        "qd":             qd_a[mask],
                        "current_clean":  cur_a[mask].copy(),
                        "current":        cur_a[mask].copy(),
                        "task":           task,
                        "is_anomaly":     is_anom,
                        "source":         key,
                    })
    return all_cycles

def precompute_torques(cycles):
    for cyc in cycles:
        payload = TASK_PAYLOAD[cyc["task"]]
        q_a = cyc["q"]
        N   = len(q_a)
        idx = list(range(0, N, SUBSAMPLE))
        tau_g_arr = np.zeros((len(idx), 6))
        for ti, t in enumerate(idx):
            tau_g_arr[ti] = gravity_torque(q_a[t], payload_mass=payload)
        cyc["tau_g_cached"] = tau_g_arr
        cyc["sub_idx"]      = idx

# Noise injection
def inject_noise(cycles_to_corrupt, sigma_signal_per_joint, snr_db, seed):
    rng = np.random.default_rng(seed)
    sigma_noise = sigma_signal_per_joint * (10 ** (-snr_db / 20.0))
    for cyc in cycles_to_corrupt:
        clean = cyc["current_clean"]
        noise = rng.normal(loc=0.0, scale=sigma_noise[np.newaxis, :], size=clean.shape)
        cyc["current"] = clean + noise

def reset_noise(cycles_to_reset):
    for cyc in cycles_to_reset:
        cyc["current"] = cyc["current_clean"].copy()

def compute_signal_sigma_per_joint(healthy_cycles):
    all_current = np.concatenate([c["current_clean"] for c in healthy_cycles], axis=0)
    return all_current.std(axis=0)

# PSR fit and 50-dim PSR features
def fit_psr_M4(train_cycles):
    Phi = [[] for _ in range(6)]
    I   = [[] for _ in range(6)]
    for cyc in train_cycles:
        qd_a = cyc["qd"]; cur = cyc["current"]
        tau_g = cyc["tau_g_cached"]
        for ti, t in enumerate(cyc["sub_idx"]):
            for j in range(6):
                qdd_j = ((qd_a[t+1, j] - qd_a[t-1, j]) * RATE / 2.0
                         if 0 < t < len(qd_a) - 1 else 0.0)
                Phi[j].append([tau_g[ti, j], qd_a[t, j],
                               np.sign(qd_a[t, j]), qdd_j, 1.0])
                I[j].append(cur[t, j])
    Phi = [np.array(p) for p in Phi]
    I   = [np.array(i) for i in I]
    w = [np.linalg.lstsq(Phi[j], I[j], rcond=None)[0] for j in range(6)]
    return w

def compute_residuals(cyc, psr_w):
    qd_a = cyc["qd"]; cur = cyc["current"]
    tau_g = cyc["tau_g_cached"]
    sub_idx = cyc["sub_idx"]
    n_sub = len(sub_idx)
    res = np.zeros((n_sub, 6))
    gr  = np.zeros((n_sub, 6))
    for ti, t in enumerate(sub_idx):
        for j in range(6):
            qdd_j = ((qd_a[t+1, j] - qd_a[t-1, j]) * RATE / 2.0
                     if 0 < t < len(qd_a) - 1 else 0.0)
            phi = np.array([tau_g[ti, j], qd_a[t, j], np.sign(qd_a[t, j]), qdd_j, 1.0])
            w = psr_w[j]
            res[ti, j] = cur[t, j] - phi @ w
            gr[ti, j]  = cur[t, j] - (w[0] * tau_g[ti, j] + w[4] * 1.0)
    return res, gr

def extract_50d_psr_features(res, gr):
    f = {}
    for j in range(6):
        r = res[:, j]; g = gr[:, j]
        f[f"J{j}_resid_mean"]     = float(r.mean())
        f[f"J{j}_resid_std"]      = float(r.std())
        f[f"J{j}_resid_rms"]      = float(np.sqrt(np.mean(r**2)))
        f[f"J{j}_resid_max"]      = float(np.abs(r).max())
        f[f"J{j}_resid_skew"]     = float(sst.skew(r))
        f[f"J{j}_resid_kurtosis"] = float(sst.kurtosis(r))
        f[f"J{j}_grav_resid_std"] = float(g.std())
        f[f"J{j}_grav_resid_rms"] = float(np.sqrt(np.mean(g**2)))
    f["total_resid_rms"] = float(np.sqrt(np.mean(res**2)))
    f["J1J2_resid_corr"] = float(np.corrcoef(res[:, 1], res[:, 2])[0, 1] if len(res) > 2 else 0.0)
    return f

PSR_FEAT_COLS_50 = (
    [f"J{j}_{s}" for j in range(6)
     for s in ["resid_mean", "resid_std", "resid_rms", "resid_max",
               "resid_skew", "resid_kurtosis",
               "grav_resid_std", "grav_resid_rms"]]
    + ["total_resid_rms", "J1J2_resid_corr"]
)

def extract_features_psr(cycles, psr_w):
    rows = []
    for cyc in cycles:
        res, gr = compute_residuals(cyc, psr_w)
        feats = extract_50d_psr_features(res, gr)
        feats["task"] = cyc["task"]
        feats["is_anomaly"] = cyc["is_anomaly"]
        rows.append(feats)
    return pd.DataFrame(rows)

# Raw Z-Score: 19-dim feature set
RAW_COLS_19 = (
    [f"J{j}_{s}" for j in range(6) for s in ["raw_mean", "raw_std", "raw_rms"]]
    + ["total_raw_rms"]
)

def extract_19d_raw_features(cyc):
    """19-dim raw feature extractor:
        cur = cyc['current']
        idx = list(range(0, len(cur), SUBSAMPLE))
        c   = cur[idx]
        per joint: raw_mean, raw_std, raw_rms
        plus:      total_raw_rms (across ALL joints, ALL subsampled timesteps)
    """
    cur = cyc["current"]
    idx = list(range(0, len(cur), SUBSAMPLE))
    c   = cur[idx]
    f   = {}
    for j in range(6):
        f[f"J{j}_raw_mean"] = float(c[:, j].mean())
        f[f"J{j}_raw_std"]  = float(c[:, j].std())
        f[f"J{j}_raw_rms"]  = float(np.sqrt(np.mean(c[:, j] ** 2)))
    f["total_raw_rms"] = float(np.sqrt(np.mean(c ** 2)))
    return f

def extract_features_raw(cycles):
    rows = []
    for cyc in cycles:
        feats = extract_19d_raw_features(cyc)
        feats["task"] = cyc["task"]
        feats["is_anomaly"] = cyc["is_anomaly"]
        rows.append(feats)
    return pd.DataFrame(rows)

def zscore_scores(Xtr, Xte):
    mu = Xtr.mean(0); sg = Xtr.std(0) + 1e-8
    return np.abs((Xte - mu) / sg).mean(1)

# Hybrid LOTO protocol
HYBRID_TRAINING = {
    "T1": ["T2", "T3"],
    "T2": ["T1", "T3"],
    "T3": ["T1", "T2"],
    "T4": ["T1", "T2", "T3"],
}

def run_loto_psr_zscore(all_cycles, test_task):
    tr_task_list = HYBRID_TRAINING[test_task]
    tr_cycs = [c for c in all_cycles if c["task"] in tr_task_list]
    te_cycs = [c for c in all_cycles if c["task"] == test_task]
    tr_healthy = [c for c in tr_cycs if c["is_anomaly"] == 0]
    psr_w = fit_psr_M4(tr_healthy)
    df_tr = extract_features_psr(tr_healthy, psr_w)
    df_te = extract_features_psr(te_cycs, psr_w)
    Xtr = df_tr[PSR_FEAT_COLS_50].values
    Xte = df_te[PSR_FEAT_COLS_50].values
    if not np.isfinite(Xtr).all() or not np.isfinite(Xte).all():
        col_mean = np.nanmean(Xtr, axis=0)
        col_mean = np.where(np.isfinite(col_mean), col_mean, 0.0)
        Xtr = np.where(np.isfinite(Xtr), Xtr, col_mean)
        Xte = np.where(np.isfinite(Xte), Xte, col_mean)
    scores = zscore_scores(Xtr, Xte)
    y_te = df_te["is_anomaly"].values
    return roc_auc_score(y_te, scores)

def run_loto_raw_zscore(all_cycles, test_task):
    tr_task_list = HYBRID_TRAINING[test_task]
    tr_cycs = [c for c in all_cycles if c["task"] in tr_task_list]
    te_cycs = [c for c in all_cycles if c["task"] == test_task]
    tr_healthy = [c for c in tr_cycs if c["is_anomaly"] == 0]
    df_tr = extract_features_raw(tr_healthy)
    df_te = extract_features_raw(te_cycs)
    Xtr = df_tr[RAW_COLS_19].values
    Xte = df_te[RAW_COLS_19].values
    if not np.isfinite(Xtr).all() or not np.isfinite(Xte).all():
        col_mean = np.nanmean(Xtr, axis=0)
        col_mean = np.where(np.isfinite(col_mean), col_mean, 0.0)
        Xtr = np.where(np.isfinite(Xtr), Xtr, col_mean)
        Xte = np.where(np.isfinite(Xte), Xte, col_mean)
    scores = zscore_scores(Xtr, Xte)
    y_te = df_te["is_anomaly"].values
    return roc_auc_score(y_te, scores)

# MAIN
print("=" * 80)
print("NB17 -- PSR Z-Score vs Raw Z-Score noise robustness (R1.2)")
print("=" * 80)

print("\n[Step 1] Loading HDF5 data...")
t0 = time.perf_counter()
all_cycles = load_all_cycles()
print(f"  {len(all_cycles)} cycles loaded in {time.perf_counter()-t0:.1f}s")
for t in ["T1", "T2", "T3", "T4"]:
    nh = sum(1 for c in all_cycles if c["task"] == t and c["is_anomaly"] == 0)
    na = sum(1 for c in all_cycles if c["task"] == t and c["is_anomaly"] == 1)
    print(f"    {t}: {nh} healthy, {na} anomaly")

print("\n[Step 2] Precomputing gravity torques...")
t0 = time.perf_counter()
precompute_torques(all_cycles)
print(f"  Done in {time.perf_counter()-t0:.1f}s")

# Step 3a: ANCHOR
print("\n" + "=" * 80)
print("[Step 3a] ANCHOR: Reproduce V5 Table 4 for both Z-Score detectors")
print("=" * 80)

reset_noise(all_cycles)
anchor_results = []
all_match = True

for test_task in ["T1", "T2", "T3", "T4"]:
    t0 = time.perf_counter()
    psr_auroc = run_loto_psr_zscore(all_cycles, test_task)
    raw_auroc = run_loto_raw_zscore(all_cycles, test_task)
    elapsed = time.perf_counter() - t0
    print(f"\n  Fold {test_task}  ({elapsed:.1f}s):")
    for det, auroc in [("PSR-ZScore", psr_auroc), ("Raw-ZScore", raw_auroc)]:
        v5 = V5_TABLE4[det][test_task]
        delta = auroc - v5
        status = "MATCH" if abs(delta) < ANCHOR_TOL else ("CLOSE" if abs(delta) < 2*ANCHOR_TOL else "MISMATCH")
        if status != "MATCH":
            all_match = False
        print(f"    {det:12s}  AUROC = {auroc:.4f}   V5={v5:.3f}   "
              f"diff={delta:+.4f}   [{status}]")
        anchor_results.append({"fold": test_task, "detector": det, "auroc": auroc,
                                "v5_table4": v5, "delta": delta, "status": status})

print()
if all_match:
    print("ANCHOR PASSED. Both detectors reproduce V5 Table 4 within +/-0.005 "
          "across all 4 folds. Proceeding to noise sweep.")
else:
    print("ANCHOR FAILED. ABORTING noise sweep.")
    import sys
    sys.exit(1)

# Save clean baselines
clean_baseline = {"PSR-ZScore": {}, "Raw-ZScore": {}}
for r in anchor_results:
    clean_baseline[r["detector"]][r["fold"]] = r["auroc"]

# Step 3b: NOISE SWEEP
print("\n" + "=" * 80)
print("[Step 3b] NOISE SWEEP -- AWGN at SNR in {-10, -5, 0, +5, +10} dB")
print("=" * 80)

SNR_LEVELS_DB = [-10, -5, 0, 5, 10]
NOISE_SEEDS   = [42, 7, 123]
results = []

# Clean baseline rows
for det, folds in clean_baseline.items():
    for fold, auroc in folds.items():
        results.append({
            "experiment": "clean_anchor",
            "snr_db": np.inf, "seed": -1,
            "fold": fold, "detector": det, "auroc": auroc,
        })

for snr_db in SNR_LEVELS_DB:
    print(f"\n  SNR = {snr_db:+d} dB:")

    for seed in NOISE_SEEDS:
        for test_task in ["T1", "T2", "T3", "T4"]:
            reset_noise(all_cycles)
            tr_task_list = HYBRID_TRAINING[test_task]
            tr_healthy = [c for c in all_cycles
                          if c["task"] in tr_task_list and c["is_anomaly"] == 0]
            te_cycs = [c for c in all_cycles if c["task"] == test_task]
            sigma_signal = compute_signal_sigma_per_joint(tr_healthy)
            inject_noise(te_cycs, sigma_signal, snr_db, seed)

            psr_auroc = run_loto_psr_zscore(all_cycles, test_task)
            raw_auroc = run_loto_raw_zscore(all_cycles, test_task)
            results.append({"experiment": "expA_clean_train_noisy_test",
                            "snr_db": float(snr_db), "seed": seed, "fold": test_task,
                            "detector": "PSR-ZScore", "auroc": psr_auroc})
            results.append({"experiment": "expA_clean_train_noisy_test",
                            "snr_db": float(snr_db), "seed": seed, "fold": test_task,
                            "detector": "Raw-ZScore", "auroc": raw_auroc})

    for seed in NOISE_SEEDS:
        for test_task in ["T1", "T2", "T3", "T4"]:
            reset_noise(all_cycles)
            tr_task_list = HYBRID_TRAINING[test_task]
            tr_cycs = [c for c in all_cycles if c["task"] in tr_task_list]
            tr_healthy = [c for c in tr_cycs if c["is_anomaly"] == 0]
            te_cycs = [c for c in all_cycles if c["task"] == test_task]
            sigma_signal = compute_signal_sigma_per_joint(tr_healthy)
            inject_noise(tr_cycs, sigma_signal, snr_db, seed)
            inject_noise(te_cycs, sigma_signal, snr_db, seed + 1000)

            psr_auroc = run_loto_psr_zscore(all_cycles, test_task)
            raw_auroc = run_loto_raw_zscore(all_cycles, test_task)
            results.append({"experiment": "expB_noisy_train_noisy_test",
                            "snr_db": float(snr_db), "seed": seed, "fold": test_task,
                            "detector": "PSR-ZScore", "auroc": psr_auroc})
            results.append({"experiment": "expB_noisy_train_noisy_test",
                            "snr_db": float(snr_db), "seed": seed, "fold": test_task,
                            "detector": "Raw-ZScore", "auroc": raw_auroc})

    # Per-SNR summary
    df_tmp = pd.DataFrame([r for r in results if r["snr_db"] == float(snr_db)])
    for exp in ["expA_clean_train_noisy_test", "expB_noisy_train_noisy_test"]:
        df_e = df_tmp[df_tmp["experiment"] == exp]
        if len(df_e) == 0:
            continue
        for det in ["PSR-ZScore", "Raw-ZScore"]:
            df_d = df_e[df_e["detector"] == det]
            sm = df_d.groupby("seed")["auroc"].mean()
            exp_label = "A" if "expA" in exp else "B"
            print(f"    EXP {exp_label}  {det:11s}: mean = {sm.mean():.4f} +/- {sm.std():.4f}")

reset_noise(all_cycles)

# Save and summary
df_results = pd.DataFrame(results)
df_results.to_csv(OUT_CSV, index=False)
print(f"\nWrote: {OUT_CSV}  ({len(df_results)} rows)")

print("\n" + "=" * 80)
print("SUMMARY -- Mean AUROC across 4 folds and 3 seeds, by SNR and detector")
print("=" * 80)
print(f"\n{'SNR (dB)':>10s}  {'EXP':>4s}  {'PSR-ZScore':>14s}  {'Raw-ZScore':>14s}  {'Δ (PSR-Raw)':>14s}")
# Clean row
psr_mean = np.mean([clean_baseline["PSR-ZScore"][f] for f in ["T1","T2","T3","T4"]])
raw_mean = np.mean([clean_baseline["Raw-ZScore"][f] for f in ["T1","T2","T3","T4"]])
print(f"{'+inf':>10s}  {'-':>4s}  {psr_mean:>14.4f}  {raw_mean:>14.4f}  {psr_mean-raw_mean:>+14.4f}  (clean baseline)")

for exp in ["expA_clean_train_noisy_test", "expB_noisy_train_noisy_test"]:
    exp_short = "A" if "expA" in exp else "B"
    for snr_db in SNR_LEVELS_DB:
        psr_sub = df_results[(df_results["experiment"] == exp) &
                             (df_results["snr_db"] == float(snr_db)) &
                             (df_results["detector"] == "PSR-ZScore")]
        raw_sub = df_results[(df_results["experiment"] == exp) &
                             (df_results["snr_db"] == float(snr_db)) &
                             (df_results["detector"] == "Raw-ZScore")]
        psr_m = psr_sub.groupby("seed")["auroc"].mean().mean()
        raw_m = raw_sub.groupby("seed")["auroc"].mean().mean()
        print(f"{snr_db:+10d}  {exp_short:>4s}  {psr_m:>14.4f}  {raw_m:>14.4f}  {psr_m-raw_m:>+14.4f}")

print("\n" + "=" * 80)
print("Done.")
print("=" * 80)